# 16 — SinhalaBERTo (Sinhala-only)

**The second monolingual Sinhala checkpoint from `model-research.md` §4**, and the cheapest run in
the roster — a small RoBERTa, trained on the `sinhala` track only.

It exists here as the control for notebook 15: if both Sinhala-only models land well below the
multilingual models on the `sinhala` cell, that is evidence about *monolingual Sinhala checkpoints
at this data scale*, not about one particular checkpoint. One model landing low is an anecdote;
two is a finding.

Its Sinhala fertility (3.40) sits between SinBERT-large (4.26) and XLM-R (1.81).

**Tokenizer fertility** (tokens per word, measured on 300 dev tickets per language):

| english | sinhala | singlish | tamil | tamilish |
|---|---|---|---|---|
| 1.41 | 3.4 | 2.16 | 7.15 | 2.63 |

Also the fastest to iterate on — use it to shake out config problems before spending an hour on a large model.

---

**Protocol.** Fit on `train` (8,500 ids), select the epoch on **dev** `negative_f1`. Test is not
opened here — only the winner of `30_encoder_leaderboard.ipynb` is refit on train+dev and scored
on test. Training code is `swiftbench.train_encoder`, shared with the other five notebooks so the
numbers land in one table.

In [1]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
import swiftbench as sb
from swiftbench import config, metrics, splits, train_encoder as te

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

AUTHOR = "sithija"
print("split sha:", splits.sha(), "| device:", te.device())

split sha: e7b5934392cd | device: mps


## Baseline to beat

The classical champion from `10_final_test_eval.ipynb`, on the same dev split. An encoder that
does not clear this is not worth its serving cost.

In [2]:
CLASSICAL_DEV = 0.6144      # tfidf-svm / class_weight / multi, pooled dev negative_f1
CLASSICAL_TEST = 0.4572     # same model, pooled test

runs = sb.results.load_all("dev")
if not runs.empty and "family" in runs.columns:
    done = runs[(runs.task == "sentiment") & (runs.family == "encoder")]
    if not done.empty:
        display(done[["model", "arm", "eval_lang", "headline", "best_epoch", "train_seconds"]]
                .sort_values("headline", ascending=False))
print(f"classical dev  negative_f1 {CLASSICAL_DEV:.4f}")
print(f"classical test negative_f1 {CLASSICAL_TEST:.4f}")

classical dev  negative_f1 0.6144
classical test negative_f1 0.4572


## Fine-tune

`SMOKE = True` runs a 1,200-row sanity pass in about a minute. Set it to `False` for the real
run (this one is cheap — Sinhala track only).

In [3]:
SMOKE = True          # <- set False for the real run

EPOCHS = 3
BATCH_SIZE = 32
LR = 2e-5
ARM = "class_weight"  # weighted loss; `ros` and `none` are the other two arms

run = te.run(
    task="sentiment",
    model="sinhalaberto",
    train_langs=["sinhala"], eval_lang="sinhala",
    arm=ARM,
    portion="dev",
    fit_portion="train",
    epochs=1 if SMOKE else EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    subsample=1200 if SMOKE else None,
    author=AUTHOR,
    save=not SMOKE,
)
run.scores

model.safetensors: reconstructing file:   0%|          |  0.00B /  334MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: keshan/SinhalaBERTo
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


sinhalaberto (keshan/SinhalaBERTo)  device=mps  max_length=256
  train 1,200 rows (was 1,200, arm=class_weight)   eval 1,200 rows   38 steps


    epoch 1/1  step     0  loss 0.6801


  epoch 1: train_loss 0.6408  negative_f1 0.1836  acc 0.8592


  best epoch 1: negative_f1 0.1836   [0.5 min]


{'accuracy': 0.8591666666666666,
 'macro_f1': 0.5532557478670606,
 'macro_precision': 0.5452780636241391,
 'macro_recall': 0.6119505494505495,
 'weighted_f1': 0.8884330687673337,
 'n': 1200,
 'negative_f1': 0.18357487922705315,
 'negative_precision': 0.12582781456953643,
 'negative_recall': 0.3392857142857143,
 'n_negative_true': 56,
 'n_negative_pred': 151,
 'headline_metric': 'negative_f1',
 'headline': 0.18357487922705315,
 'n_train': 1200,
 'n_train_before_resample': 1200,
 'best_epoch': 1,
 'epochs': 1,
 'lr': 2e-05,
 'batch_size': 32,
 'max_length': 256,
 'train_seconds': 27.3,
 'hf_name': 'keshan/SinhalaBERTo',
 'fit_portion': 'train',
 'device': 'mps'}

In [4]:
display(run.history)
print(f"selected epoch {run.scores['best_epoch']} of {run.scores['epochs']}  "
      f"({run.scores['train_seconds']/60:.1f} min on {run.scores['device']})")

,epoch,train_loss,accuracy,macro_f1,macro_precision,macro_recall,weighted_f1,n,negative_f1,negative_precision,negative_recall,n_negative_true,n_negative_pred,headline
0,1,0.6408,0.8592,0.5533,0.5453,0.6120,0.8884,1200,0.1836,0.1258,0.3393,56,151,0.1836


selected epoch 1 of 1  (0.5 min on mps)


## Where it fails

Selection metric alone hides the operating point. At 95%+ Neutral, a model can post a healthy
Negative-F1 while its precision makes escalation unusable — the bake-off's encoders all sat at
0.12-0.20 precision against 0.77-0.90 recall.

In [5]:
ev = run.eval_frame.copy()
ev["pred"] = run.predictions
ev["p_negative"] = run.scores_positive

print("confusion (rows = truth)")
labels, cm = metrics.confusion(ev.sentiment, ev.pred, "sentiment")
display(pd.DataFrame(cm, index=labels, columns=labels))

print(f"\nprecision {run.scores['negative_precision']:.4f}   "
      f"recall {run.scores['negative_recall']:.4f}   "
      f"negative_f1 {run.scores['headline']:.4f}")

confusion (rows = truth)


,Neutral,Negative
Neutral,1012,132
Negative,37,19



precision 0.1258   recall 0.3393   negative_f1 0.1836


In [6]:
# Per-language breakdown — this model only sees sinhala, shown for consistency.
rows = []
for lang in ['sinhala']:
    m = (ev.language == lang)
    if not m.any():
        continue
    s = metrics.score(ev.sentiment[m], ev.pred[m], "sentiment")
    rows.append({"language": lang, **{k: v for k, v in s.items() if not isinstance(v, str)}})
per_lang = pd.DataFrame(rows)
display(per_lang[["language", "negative_f1", "negative_precision", "negative_recall",
                  "accuracy", "n_negative_true"]])

,language,negative_f1,negative_precision,negative_recall,accuracy,n_negative_true
0,sinhala,0.1836,0.1258,0.3393,0.8592,56


In [7]:
# False negatives -- angry customers routed to an auto-reply. These are the expensive errors.
missed = ev[(ev.sentiment == "Negative") & (ev.pred != "Negative")]
print(f"{len(missed)} missed Negative rows of {(ev.sentiment == 'Negative').sum()}")
display(missed.sort_values("p_negative", ascending=False)[["language", "text", "p_negative"]].head(12))

37 missed Negative rows of 56


,language,text,p_negative
703,sinhala,ඇප් එකේ list වෙච්ච direct debit payment එකක් ම...,0.4990
1029,sinhala,"හලෝ, online shopping කරද්දි මගේ කාඩ් payment එ...",0.4917
222,sinhala,මට බය හිතෙනවා! මගේ කාඩ් එක නැති වුණා! උදව් කරන්න!,0.4873
304,sinhala,පැයකට කලින් මම කරපු top up එකේ මොකක් හරි වැරදි...,0.4794
505,sinhala,"උදව් කරන්න! මගේ සල්ලි missing, cheque එකක් dep...",0.4762
694,sinhala,App එකේ මගේ නෙවෙයි direct debit එකක් දකිනවා,0.4762
547,sinhala,මම කරන්නේ නැති payment එකක් මගේ ඇප් එකේ පේනවා.,0.4747
887,sinhala,මට හරිම embarrassing වුණා. මගේ payment එක clea...,0.4686
1321,sinhala,මම කරන්නේ නැති cash withdrawal එකක් වෙලා.,0.4660
789,sinhala,මට උදව් කරන්න. මගේ phone එක නැති වුණා නැත්තම් ...,0.4639


## Verdict

Fill this in after the real run.

- Beats classical dev (0.6144) — yes / no, and by how much against the CI width of ~0.15.
- Where it wins and loses per language, especially the `sinhala` cell.
- Whether precision is high enough for escalation, or whether it needs the lexicon layer from
  `20_technique_lexicon_correction.ipynb`.

Then record it in `30_encoder_leaderboard.ipynb`.